# Create David and Elaine Potter Foundation Awards

Creates OpenAlex award rows from The David and Elaine Potter Foundation's official grant-recipient sources.

**Awarding body:** David and Elaine Potter Foundation - F4320314720 (GB; no ROR or DOI in OpenAlex)

**Sources:**
- https://www.potterfoundation.com/grant_recipients.html
- https://www.potterfoundation.com/downloads/360giving_Potter_Foundation_data_2013-2024.xlsx

**Source method:** official bulk workbook plus static HTML tables. The linked 360Giving workbook is used for 2013-2024 because it carries native identifiers, descriptions, award dates, recipient identifiers, programme labels, and GBP amounts. The official grant-recipient HTML tables are used for 2000-2012 legacy rows.

**Local validation on 2026-05-28:** 561 rows, 561 unique native IDs, years 2000-2024, 100% title/recipient/amount/currency/year/program/landing-page coverage, 275/561 description coverage from the structured workbook, and total GBP 23,657,103.

**S3 staging path:** `s3a://openalex-ingest/awards/potter_foundation/potter_foundation_grants.parquet`

**Schema choices:**
- `funder_award_id` uses official 360Giving identifiers for 2013-2024 and stable synthetic IDs for older HTML rows.
- `funding_type` is `research` when the programme is Research, otherwise `grant`.
- `funder_scheme` is the source programme/category.
- `start_date` uses the exact award date when the workbook provides one, otherwise January 1 of the source award year.
- `lead_investigator` is organization-only using the recipient organization name and normalized recipient country when available.
- `amount` and `currency` are populated from official GBP grant commitments; no amount waiver is needed.


## Step 1: Create Raw Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.potter_foundation_grants_raw
USING delta
AS
SELECT
    *,
    current_timestamp() AS databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/potter_foundation/potter_foundation_grants.parquet`;


In [ ]:
%sql
-- Check raw row count. Local validation on 2026-05-28 found 561 rows.
SELECT COUNT(*) AS total_potter_grants
FROM openalex.awards.potter_foundation_grants_raw;


In [ ]:
%sql
-- Verify actual column names before transform logic references them.
DESCRIBE openalex.awards.potter_foundation_grants_raw;


In [ ]:
%sql
-- Sample raw Potter Foundation data.
SELECT
    funder_award_id,
    source_record_type,
    display_name,
    recipient_name,
    amount,
    currency,
    award_date,
    award_year,
    program_title,
    recipient_country,
    landing_page_url
FROM openalex.awards.potter_foundation_grants_raw
LIMIT 10;


## Step 1.5: Inspect Raw Data, Money, Dates, and Native Keys

In [ ]:
%sql
-- Money-field scan per runbook Step 1.5. Do not wrap DESCRIBE in a subquery.
SELECT column_name
FROM openalex.information_schema.columns
WHERE table_schema = 'awards'
  AND table_name = 'potter_foundation_grants_raw'
  AND LOWER(column_name) RLIKE 'amount|amt|total|funding|award|grant|currency|gbp|pound|value|budget|cost'
ORDER BY ordinal_position;


In [ ]:
%sql
-- Confirm amount/date/source coverage before mapping.
SELECT
    COUNT(*) AS total,
    COUNT(amount) AS rows_with_amount,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_with_amount,
    COUNT(currency) AS rows_with_currency,
    ROUND(try_divide(COUNT(currency), COUNT(*)) * 100.0, 1) AS pct_with_currency,
    COUNT(award_date) AS rows_with_exact_award_date,
    COUNT(award_year) AS rows_with_award_year,
    ROUND(SUM(TRY_CAST(amount AS DOUBLE)), 0) AS total_gbp,
    ROUND(MIN(TRY_CAST(amount AS DOUBLE)), 0) AS min_gbp,
    ROUND(MAX(TRY_CAST(amount AS DOUBLE)), 0) AS max_gbp,
    MIN(TRY_CAST(award_year AS INT)) AS min_award_year,
    MAX(TRY_CAST(award_year AS INT)) AS max_award_year
FROM openalex.awards.potter_foundation_grants_raw;


In [ ]:
%sql
-- Native-key inspection. 360Giving rows use official identifiers; legacy HTML rows use deterministic synthetic IDs.
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT funder_award_id) AS distinct_funder_award_ids,
    COUNT(DISTINCT source_record_id) AS distinct_source_record_ids,
    COUNT(*) - COUNT(DISTINCT funder_award_id) AS duplicate_funder_award_ids
FROM openalex.awards.potter_foundation_grants_raw;


In [ ]:
%sql
-- Source type and year distribution.
SELECT
    source_record_type,
    award_year,
    COUNT(*) AS rows,
    ROUND(SUM(TRY_CAST(amount AS DOUBLE)), 0) AS total_gbp
FROM openalex.awards.potter_foundation_grants_raw
GROUP BY source_record_type, award_year
ORDER BY TRY_CAST(award_year AS INT), source_record_type;


In [ ]:
%sql
-- Programme distribution.
SELECT
    program_title,
    COUNT(*) AS rows,
    ROUND(SUM(TRY_CAST(amount AS DOUBLE)), 0) AS total_gbp,
    ROUND(try_divide(COUNT(description), COUNT(*)) * 100.0, 1) AS pct_with_description
FROM openalex.awards.potter_foundation_grants_raw
GROUP BY program_title
ORDER BY rows DESC;


In [ ]:
%sql
-- Recipient country coverage.
SELECT
    COUNT(*) AS total,
    COUNT(recipient_country) AS has_recipient_country,
    ROUND(try_divide(COUNT(recipient_country), COUNT(*)) * 100.0, 1) AS pct_recipient_country,
    collect_set(recipient_country) AS recipient_countries
FROM openalex.awards.potter_foundation_grants_raw;


## Step 1.6: Funder Existence Fail-Fast

Runbook Step 2.2.4 mandatory check. Must return exactly 1 row for David and Elaine Potter Foundation.

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi
FROM openalex.common.funder
WHERE funder_id = 4320314720;


In [ ]:
%sql
-- Fail-fast count form. Expected: exactly one row.
SELECT COUNT(*) AS funder_rows
FROM openalex.common.funder
WHERE funder_id = 4320314720;


## Step 2: Transform to OpenAlex Awards Schema

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.potter_foundation_awards
USING delta
AS
WITH
potter_funder AS (
    SELECT
        funder_id,
        display_name,
        ror_id,
        doi
    FROM openalex.common.funder
    WHERE funder_id = 4320314720
),

raw_typed AS (
    SELECT
        *,
        LOWER(TRIM(funder_award_id)) AS native_award_id,
        TRY_CAST(amount AS DOUBLE) AS parsed_amount,
        CASE WHEN TRY_CAST(amount AS DOUBLE) IS NOT NULL THEN currency ELSE NULL END AS parsed_currency,
        TRY_CAST(award_year AS INT) AS parsed_year,
        TRY_CAST(duration_months AS INT) AS parsed_duration_months,
        COALESCE(
            TRY_TO_DATE(award_date, 'yyyy-MM-dd'),
            CASE
                WHEN TRY_CAST(award_year AS INT) BETWEEN 1900 AND YEAR(current_date()) + 1
                THEN TRY_TO_DATE(CONCAT(award_year, '-01-01'), 'yyyy-MM-dd')
                ELSE NULL
            END
        ) AS parsed_start_date
    FROM openalex.awards.potter_foundation_grants_raw
    WHERE funder_award_id IS NOT NULL
      AND TRIM(funder_award_id) != ''
      AND display_name IS NOT NULL
      AND TRIM(display_name) != ''
),

raw_prepared AS (
    SELECT
        *,
        CASE
            WHEN parsed_start_date IS NOT NULL AND parsed_duration_months IS NOT NULL AND parsed_duration_months > 0
            THEN date_sub(add_months(parsed_start_date, parsed_duration_months), 1)
            ELSE NULL
        END AS parsed_end_date
    FROM raw_typed
),

awards_transformed AS (
    SELECT
        abs(xxhash64(CONCAT(f.funder_id, ':', g.native_award_id))) % 9000000000 AS id,

        TRIM(g.display_name) AS display_name,

        CASE
            WHEN g.description IS NULL OR TRIM(g.description) = '' THEN NULL
            ELSE TRIM(g.description)
        END AS description,

        f.funder_id,
        g.native_award_id AS funder_award_id,

        g.parsed_amount AS amount,
        g.parsed_currency AS currency,

        struct(
            CONCAT('https://openalex.org/F', f.funder_id) AS id,
            f.display_name,
            f.ror_id,
            f.doi
        ) AS funder,

        CASE
            WHEN LOWER(TRIM(g.program_title)) = 'research' THEN 'research'
            ELSE 'grant'
        END AS funding_type,

        NULLIF(TRIM(g.program_title), '') AS funder_scheme,

        'potter_foundation_grants' AS provenance,

        CASE WHEN YEAR(g.parsed_start_date) > YEAR(current_date()) + 1 THEN NULL ELSE g.parsed_start_date END AS start_date,
        CASE WHEN YEAR(g.parsed_start_date) > YEAR(current_date()) + 1 THEN NULL ELSE g.parsed_end_date END AS end_date,
        CASE WHEN COALESCE(YEAR(g.parsed_start_date), g.parsed_year) > YEAR(current_date()) + 1 THEN NULL ELSE COALESCE(YEAR(g.parsed_start_date), g.parsed_year) END AS start_year,
        CASE WHEN COALESCE(YEAR(g.parsed_start_date), g.parsed_year) > YEAR(current_date()) + 1 THEN NULL ELSE YEAR(g.parsed_end_date) END AS end_year,

        struct(
            CAST(NULL AS STRING) AS given_name,
            CAST(NULL AS STRING) AS family_name,
            CAST(NULL AS STRING) AS orcid,
            g.parsed_start_date AS role_start,
            struct(
                NULLIF(TRIM(g.recipient_name), '') AS name,
                NULLIF(TRIM(g.recipient_country), '') AS country,
                CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) AS ids
            ) AS affiliation
        ) AS lead_investigator,

        CAST(NULL AS STRUCT<
            given_name:STRING,
            family_name:STRING,
            orcid:STRING,
            role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >) AS co_lead_investigator,

        CAST(NULL AS ARRAY<STRUCT<
            given_name:STRING,
            family_name:STRING,
            orcid:STRING,
            role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >>) AS investigators,

        NULLIF(TRIM(g.landing_page_url), '') AS landing_page_url,
        CAST(NULL AS STRING) AS doi,

        CONCAT('https://api.openalex.org/works?filter=awards.id:G', abs(xxhash64(CONCAT(f.funder_id, ':', g.native_award_id))) % 9000000000) AS works_api_url,

        current_timestamp() AS created_date,
        current_timestamp() AS updated_date

    FROM raw_prepared g
    CROSS JOIN potter_funder f
)

SELECT * FROM awards_transformed;


## Step 3: Delete Previous Source Rows and Insert Priority 168

In [ ]:
%sql
-- Remove previous Potter Foundation data before inserting fresh data.
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'potter_foundation_grants' AND priority = 168;

-- Insert into openalex_awards_raw with exact OpenAlex awards schema plus priority.
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id,
    display_name,
    description,
    funder_id,
    funder_award_id,
    amount,
    currency,
    funder,
    funding_type,
    funder_scheme,
    provenance,
    start_date,
    end_date,
    start_year,
    end_year,
    lead_investigator,
    co_lead_investigator,
    investigators,
    landing_page_url,
    doi,
    works_api_url,
    created_date,
    updated_date,
    168 AS priority
FROM openalex.awards.potter_foundation_awards;


## Step 6: Verification

In [ ]:
%sql
SELECT COUNT(*) AS total_potter_foundation_awards
FROM openalex.awards.potter_foundation_awards;


In [ ]:
%sql
-- Confirm the transformed table has the OpenAlex awards columns only.
DESCRIBE openalex.awards.potter_foundation_awards;


In [ ]:
%sql
-- Sample transformed awards.
SELECT
    id,
    display_name,
    funder_award_id,
    funder_scheme,
    funding_type,
    amount,
    currency,
    start_date,
    end_date,
    lead_investigator.affiliation.name AS recipient_org,
    lead_investigator.affiliation.country AS recipient_country,
    landing_page_url
FROM openalex.awards.potter_foundation_awards
LIMIT 10;


In [ ]:
%sql
-- Check OpenAlex ID and native-key uniqueness.
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT id) AS distinct_openalex_ids,
    COUNT(DISTINCT funder_award_id) AS distinct_funder_award_ids,
    COUNT(*) - COUNT(DISTINCT id) AS duplicate_openalex_ids,
    COUNT(*) - COUNT(DISTINCT funder_award_id) AS duplicate_funder_award_ids
FROM openalex.awards.potter_foundation_awards;


In [ ]:
%sql
-- Check funder distribution. Should be only David and Elaine Potter Foundation.
SELECT funder.display_name, funder_id, COUNT(*) AS rows
FROM openalex.awards.potter_foundation_awards
GROUP BY funder.display_name, funder_id
ORDER BY rows DESC;


In [ ]:
%sql
-- Data completeness.
SELECT
    COUNT(*) AS total,
    COUNT(display_name) AS has_title,
    COUNT(description) AS has_description,
    COUNT(amount) AS has_amount,
    COUNT(currency) AS has_currency,
    COUNT(start_year) AS has_start_year,
    COUNT(start_date) AS has_start_date,
    COUNT(end_date) AS has_end_date,
    COUNT(landing_page_url) AS has_url,
    COUNT(lead_investigator.affiliation.name) AS has_recipient_org,
    COUNT(lead_investigator.affiliation.country) AS has_recipient_country,
    ROUND(SUM(amount), 0) AS total_gbp,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_with_amount,
    ROUND(try_divide(COUNT(description), COUNT(*)) * 100.0, 1) AS pct_description,
    ROUND(try_divide(COUNT(lead_investigator.affiliation.country), COUNT(*)) * 100.0, 1) AS pct_recipient_country
FROM openalex.awards.potter_foundation_awards;


In [ ]:
%sql
-- Amount/currency fail-fast check for this grant pattern. Expected: 100% GBP amounts.
SELECT
    COUNT(*) AS total,
    SUM(CASE WHEN amount > 0 THEN 1 ELSE 0 END) AS rows_with_positive_amount,
    ROUND(try_divide(SUM(CASE WHEN amount > 0 THEN 1 ELSE 0 END), COUNT(*)) * 100.0, 1) AS pct_positive_amount,
    COUNT(DISTINCT currency) AS distinct_currencies,
    collect_set(currency) AS currencies,
    ROUND(MIN(amount), 0) AS min_amount,
    ROUND(MAX(amount), 0) AS max_amount,
    ROUND(AVG(CASE WHEN amount > 0 THEN amount END), 0) AS avg_positive_amount,
    ROUND(SUM(amount), 0) AS total_amount
FROM openalex.awards.potter_foundation_awards;


In [ ]:
%sql
-- Year distribution.
SELECT start_year, COUNT(*) AS rows, ROUND(SUM(amount), 0) AS total_gbp
FROM openalex.awards.potter_foundation_awards
WHERE start_year IS NOT NULL
GROUP BY start_year
ORDER BY start_year;


In [ ]:
%sql
-- Funding type and programme distribution.
SELECT funding_type, funder_scheme, COUNT(*) AS rows, ROUND(SUM(amount), 0) AS total_gbp
FROM openalex.awards.potter_foundation_awards
GROUP BY funding_type, funder_scheme
ORDER BY rows DESC;


In [ ]:
%sql
-- Verify rows inserted into the raw awards table at priority 168.
SELECT
    priority,
    provenance,
    COUNT(*) AS rows,
    COUNT(DISTINCT funder_award_id) AS distinct_funder_award_ids,
    ROUND(SUM(amount), 0) AS total_gbp
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'potter_foundation_grants' AND priority = 168
GROUP BY priority, provenance;


## Handoff Notes

Contractor has no S3 or Databricks access. Admin should:

1. Run `scripts/local/potter_foundation_to_s3.py --skip-download` without `--skip-upload` after reviewing the local parquet.
2. Run this notebook on Databricks.
3. Inspect Step 6 verification cells for row count, uniqueness, funder mapping, amount/currency coverage, and priority 168 insertion.
4. Only add scheduled job YAML after upload, notebook execution, and QA pass.
